# Chunking experiments

This notebook loads a sample arXiv abstract from the project metadata file and compares the available chunking strategies using the current processing package API.

In [1]:
import json
import sys
from pathlib import Path

project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.rag_curator.processing.chunker import ChunkManager
from src.rag_curator.processing.strategies import FixedSizeChunker, RecursiveChunker, SentenceChunker, SemanticChunker


In [2]:
metadata_path = project_root / 'data' / 'sample_arxiv_metadata.json'

with metadata_path.open('r', encoding='utf-8') as f:
    metadata = json.load(f)

print(f"Loaded {len(metadata.get('papers', []))} papers from {metadata_path}\n")
first_paper = metadata['papers'][0]
print(f"Paper: {first_paper.get('title', 'Untitled')}\n")
print(f"Abstract length: {len(first_paper.get('abstract', ''))} chars")

sample_text = first_paper.get('abstract', '')


Loaded 37 papers from /Users/aditya/Desktop/Projects/Project/production-rag/data/sample_arxiv_metadata.json

Paper: Decoding the Past: An Uncertainty-Aware Deep Learning Framework for Sex Attribution in Prehistoric Hand Stencils

Abstract length: 1903 chars


In [7]:
manager = ChunkManager()

fixed_chunks = manager.chunk(
    sample_text,
    strategy='fixed',
    chunk_size=300,
    overlap=50,
    source='sample_arxiv_metadata',
    preprocess=True,
)

print(f"Fixed chunks: {len(fixed_chunks)} Completed\n")

recursive_chunks = manager.chunk(
    sample_text,
    strategy='recursive',
    chunk_size=300,
    overlap=50,
    source='sample_arxiv_metadata',
    preprocess=True,
)

print(f"Recursive chunks: {len(recursive_chunks)} Completed\n")

sentence_chunks = manager.chunk(
    sample_text,
    strategy='sentence',
    chunk_size=300,
    overlap=50,
    source='sample_arxiv_metadata',
    preprocess=True,
)

print(f"Sentence chunks: {len(sentence_chunks)} Completed\n")

print('fixed', len(fixed_chunks))
print('recursive', len(recursive_chunks))
print('sentence', len(sentence_chunks))


2026-08-18 17:06:39 [INFO] rag_curator.processing.chunker: Chunking request: strategy=fixed, chunk_size=300, overlap=50, source=sample_arxiv_metadata, preprocess=True
2026-08-18 17:06:39 [INFO] rag_curator.processing.chunker: Starting preprocessing for text of length 1903
2026-08-18 17:06:39 [INFO] rag_curator.processing.cleaner: Starting text cleaning for 1903 characters
2026-08-18 17:06:39 [INFO] rag_curator.processing.cleaner: Text cleaning complete. Output length: 1903 characters
2026-08-18 17:06:39 [INFO] rag_curator.processing.cleaner: Removed header/footer lines: 1 -> 1
2026-08-18 17:06:39 [INFO] rag_curator.processing.chunker: Preprocessing complete. Final text length: 1903
2026-08-18 17:06:39 [INFO] rag_curator.processing.chunker: Chunking complete: produced 8 chunks for strategy=fixed
2026-08-18 17:06:39 [INFO] rag_curator.processing.chunker: Chunking request: strategy=recursive, chunk_size=300, overlap=50, source=sample_arxiv_metadata, preprocess=True
2026-08-18 17:06:39 [IN

Fixed chunks: 8 Completed

Recursive chunks: 9 Completed

Sentence chunks: 1 Completed

fixed 8
recursive 9
sentence 1


## Compare chunk boundary behavior

The next cell lets you inspect the actual chunk text returned by each strategy and helps decide which strategy fits the retrieval use case best.

In [4]:
def summarize_chunks(label, chunks):
    if not chunks:
        print(f'{label}: no chunks created')
        return

    print(f'\n{label}: {len(chunks)} chunks')
    for i, chunk in enumerate(chunks[:3]):
        print(f'  chunk {i}: chars={len(chunk.text)}, words={len(chunk.text.split())}')
        print(f'    preview={chunk.text[:120]!r}')

summarize_chunks('fixed', fixed_chunks)
summarize_chunks('recursive', recursive_chunks)
summarize_chunks('sentence', sentence_chunks)



fixed: 8 chunks
  chunk 0: chars=300, words=40
    preview='Determining the biological sex of the individuals who created Upper Paleolithic hand stencils remains a challenging prob'
  chunk 1: chars=300, words=36
    preview='troduced by image degradation. Traditional morphometric methods suffer from high structural overlap across sexes, poor c'
  chunk 2: chars=300, words=36
    preview='work for sex attribution in prehistoric hand stencils that explicitly models, propagates, and aggregates uncertainty thr'

recursive: 9 chunks
  chunk 0: chars=279, words=38
    preview='Determining the biological sex of the individuals who created Upper Paleolithic hand stencils remains a challenging prob'
  chunk 1: chars=208, words=24
    preview='nd the uncertainty introduced by image degradation Traditional morphometric methods suffer from high structural overlap '
  chunk 2: chars=262, words=31
    preview='neralizability, and subjective feature engineering This study presents an uncertainty-aw

In [5]:
for label, chunks in {
    'fixed': fixed_chunks,
    'recursive': recursive_chunks,
    'sentence': sentence_chunks,
}.items():
    print(f'\n### {label.upper()}')
    for idx, chunk in enumerate(chunks[:2]):
        print(f'Chunk {idx}:')
        print(chunk.text[:300])
        print('-' * 80)



### FIXED
Chunk 0:
Determining the biological sex of the individuals who created Upper Paleolithic hand stencils remains a challenging problem due to the absence of ground truth, population differences between contemporary and prehistoric groups, and the uncertainty introduced by image degradation. Traditional morphom
--------------------------------------------------------------------------------
Chunk 1:
troduced by image degradation. Traditional morphometric methods suffer from high structural overlap across sexes, poor cross-population generalizability, and subjective feature engineering. This study presents an uncertainty-aware deep learning framework for sex attribution in prehistoric hand stenc
--------------------------------------------------------------------------------

### RECURSIVE
Chunk 0:
Determining the biological sex of the individuals who created Upper Paleolithic hand stencils remains a challenging problem due to the absence of ground truth, population differences 